# Multi-3x Price-Volume Transfer V1

## tl;dr

本卡只检验量价因子本身：先在非 SOXL 的长期 3x ETF 上逐 underlying 验证方向与冗余，再冻结因子并在 2023–2026 SOXL 上零样本揭盲。模型与仓位控制不参与主结论。

## Context & Methods

### Key Assumptions

- 原始 OHLCV 通过 `qlib.init` 与 `qlib.data.D.features` 从配对 ETF/underlying provider 回读。
- 8 个量价压力因子按单 ETF 历史滚动归一化并滞后一完整交易日，数值越高代表未来 20 日 breakdown 压力越高。
- SOXL 完全不参与因子筛选；开发集截止 2022-12-31，外层 SOXL 从 2023-01-01 开始。
- 主样本限定为历史高位状态，基准是 price-only 等权组合；没有拟合预测模型。
- qlib `DatasetH`/`DataHandlerLP`/`QlibDataLoader` 用于数据合同，`R.start`/Recorder 保存逐行证据。

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.workflow import R

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if ROOT.as_posix() not in sys.path:
    sys.path.insert(0, ROOT.as_posix())

from src.workflow.multi_3x_price_volume_transfer import (
    EXPERIMENT,
    run_multi_3x_price_volume_transfer_card,
)

CARD = ROOT / 'research' / EXPERIMENT
print({'experiment': EXPERIMENT, 'source': 'qlib.data.D.features', 'factor_mode': 'factor_only'})

{'experiment': 'multi_3x_price_volume_transfer_v1', 'source': 'qlib.data.D.features', 'factor_mode': 'factor_only'}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Data

下面执行配对 provider 回读、点时量价因子构造、派生 qlib store 审计、DatasetH 分段、非 SOXL 跨组筛选、SOXL 零样本外评估及 Recorder 登记。

In [2]:
summary = run_multi_3x_price_volume_transfer_card(root=ROOT)
assert summary['qlib_contract']['data_api'] == ['qlib.init', 'qlib.data.D.features']
assert summary['qlib_contract']['dataset_api'] == ['DatasetH', 'DataHandlerLP', 'QlibDataLoader']
assert summary['factor_contract']['selection_uses_soxl'] is False
print({'status': summary['status'], 'recorder_id': summary['recorder']['recorder_id'], 'decision': summary['promotion_decision']})

[40579:MainThread](2026-07-19 16:53:37,350) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40579:MainThread](2026-07-19 16:53:37,353) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40579:MainThread](2026-07-19 16:53:37,354) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/multi-3x-high-chop-risk-control-v1/data/qlib/soxl')}


[40579:MainThread](2026-07-19 16:53:37,376) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40579:MainThread](2026-07-19 16:53:37,379) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

[40579:MainThread](2026-07-19 16:53:41,866) INFO - qlib.timer - [log.py:127] - Time cost: 0.168s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[40579:MainThread](2026-07-19 16:53:42,122) INFO - qlib.workflow - [exp.py:258] - Experiment 616802982604554401 starts running ...


[40579:MainThread](2026-07-19 16:53:42,214) INFO - qlib.workflow - [recorder.py:345] - Recorder 66d4bd67402e45bb92fdfbfa739ba2aa starts running under Experiment 616802982604554401 ...


[40579:MainThread](2026-07-19 16:53:42,411) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'status': 'completed', 'recorder_id': '66d4bd67402e45bb92fdfbfa739ba2aa', 'decision': 'not_retained_no_transferable_price_volume_evidence'}


## Results

In [3]:
factor_summary = pd.read_csv(CARD / 'factor_summary.csv')
group_metrics = pd.read_csv(CARD / 'group_metrics.csv')
outer_metrics = pd.read_csv(CARD / 'outer_test_metrics.csv')
display(factor_summary.sort_values(['development_gate_passed', 'median_group_auc'], ascending=[False, False]))
display(outer_metrics)
print({'eligible_groups': int(group_metrics.loc[group_metrics['eligible'], 'underlying_group'].nunique()), 'selected_factors': summary['factor_contract']['selected_factors']})

,factor,eligible_group_count,direction_consistency,median_group_auc,median_group_quintile_lift,development_gate_passed,selection_status
3,price_volume_divergence_pressure_20,14,0.785714,0.533333,0.030357,True,selected
5,signed_volume_selling_pressure_20,14,0.714286,0.522817,0.023801,True,selected
7,volume_weighted_clv_pressure_20,14,0.500000,0.530330,0.020855,False,failed_development_gate
4,realized_leverage_shortfall_pressure_60,14,0.571429,0.529665,0.028653,False,failed_development_gate
1,failed_breakout_pressure_20,14,0.571429,0.528630,0.039231,False,failed_development_gate
6,volume_sign_flip_pressure_20,14,0.285714,0.487274,-0.009891,False,failed_development_gate
2,leverage_residual_pressure_20,14,0.357143,0.485315,-0.022306,False,failed_development_gate
0,effort_result_pressure_20,14,0.142857,0.462406,-0.033274,False,failed_development_gate


,window,composite,observations,breakdown_events,auc,average_precision,prevalence,average_precision_lift,quintile_1_rate,quintile_5_rate,quintile_lift,quintile_monotonicity
0,overall,price_only,461,33,0.704758,0.125492,0.071584,1.753083,0.010753,0.141304,0.130552,1.000000
1,overall,price_volume,461,33,0.431960,0.061788,0.071584,0.863154,0.096774,0.032609,-0.064165,-0.600000
2,2023,price_only,198,17,0.722782,0.161840,0.085859,1.884957,0.000000,0.175000,0.175000,1.000000
3,2023,price_volume,198,17,0.449139,0.104281,0.085859,1.214563,0.125000,0.100000,-0.025000,-0.205196
4,2024,price_only,70,11,0.619414,0.308875,0.157143,1.965569,0.071429,0.285714,0.214286,0.615587
5,2024,price_volume,70,11,0.216487,0.105975,0.157143,0.674388,0.357143,0.000000,-0.357143,-0.820783
6,2025,price_only,122,3,0.801120,0.085575,0.024590,3.480052,0.000000,0.040000,0.040000,0.782624
7,2025,price_volume,122,3,0.584034,0.037947,0.024590,1.543169,0.000000,0.000000,0.000000,0.223607
8,2026,price_only,71,2,0.913043,0.200000,0.028169,7.100000,0.000000,0.142857,0.142857,0.707107
9,2026,price_volume,71,2,0.710145,0.099206,0.028169,3.521825,0.000000,0.071429,0.071429,0.577350


{'eligible_groups': 14, 'selected_factors': ['price_volume_divergence_pressure_20', 'signed_volume_selling_pressure_20']}


## Takeaways

In [4]:
decision = summary['decision']
failed = [name for name, gate in decision['gates'].items() if not gate['passed']]
print(f"执行内容：非SOXL跨underlying开发验证 + 2023–2026 SOXL零样本外揭盲；无预测模型和仓位控制。")
print(f"关键结果：{(CARD / 'conclusion.md').read_text(encoding='utf-8').splitlines()[4]}")
print(f"区间差异：outer_test_metrics.csv 同时列出 overall 与各年度方向，失败门槛为 {', '.join(failed) if failed else '无'}。")
print(f"晋升决定：{decision['decision']}；只决定是否保留到下一轮风险控制研究，不是交易策略晋升。")
print("风险：日频量价不能还原盘中交易顺序，ETF产品年龄与行业结构不同，breakdown标签也不是直接收益。")
print("下一步：只继续检验跨组且SOXL有增量的因子；未通过的因子停止调模型。")

执行内容：非SOXL跨underlying开发验证 + 2023–2026 SOXL零样本外揭盲；无预测模型和仓位控制。
关键结果：- 保留因子：price_volume_divergence_pressure_20, signed_volume_selling_pressure_20。
区间差异：outer_test_metrics.csv 同时列出 overall 与各年度方向，失败门槛为 outer_auc, outer_ap_lift_increment, outer_quintile_lift_increment, annual_direction_consistency。
晋升决定：not_retained_no_transferable_price_volume_evidence；只决定是否保留到下一轮风险控制研究，不是交易策略晋升。
风险：日频量价不能还原盘中交易顺序，ETF产品年龄与行业结构不同，breakdown标签也不是直接收益。
下一步：只继续检验跨组且SOXL有增量的因子；未通过的因子停止调模型。
